# EmPath · Data cleaning and quality checks

This notebook inspects the first cleaning pass for **75025 plus 250 m of context**. Raw snapshots are preserved. Sidewalk polygons are not yet a routable graph. See `docs/resources.md` for the annotated resource guide.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
from clean_data import run
report = run()
DATA = ROOT / "data/processed/75025_buffer250m"

: 

## Cleaning accounting

Records returned from a rectangular API query may fall outside the actual buffered polygon. Those are spatially filtered, not classified as bad data. Distinct records sharing a geometry are retained and flagged for review.

In [ ]:
import pandas as pd
counts = pd.DataFrame(report["layers"]).T
counts[["raw_rows", "clean_rows", "repaired_rows", "quarantined_rows", "outside_context_rows", "duplicate_geometry_rows_retained"]]

## Geometry inspection

The overview shows cleaned sidewalk **surfaces** and trail **lines**. The detailed panel shows a small area near the study-area interior. Parcel boundaries are not paths, and address points are not confirmed entrances.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from shapely.geometry import box
sidewalks = gpd.read_file(DATA / "sidewalk_surfaces.gpkg")
trails = gpd.read_file(DATA / "trails.gpkg")
parcels = gpd.read_file(DATA / "parcels.gpkg")
core = gpd.read_file(DATA / "core_zctas.geojson").to_crs(sidewalks.crs)
center = core.geometry.unary_union.representative_point()
window = box(center.x - 500, center.y - 500, center.x + 500, center.y + 500)
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
core.boundary.plot(ax=axes[0], color="#334155", linewidth=1)
sidewalks.plot(ax=axes[0], color="#d65f35", linewidth=.2)
trails.plot(ax=axes[0], color="#187e8a", linewidth=.8)
local_parcels = parcels[parcels.intersects(window)]
local_parcels.boundary.plot(ax=axes[1], color="#c1c9cf", linewidth=.4)
sidewalks[sidewalks.intersects(window)].plot(ax=axes[1], color="#d65f35")
local_trails = trails[trails.intersects(window)]
if not local_trails.empty:
    local_trails.plot(ax=axes[1], color="#187e8a", linewidth=1)
axes[1].set_xlim(window.bounds[0], window.bounds[2])
axes[1].set_ylim(window.bounds[1], window.bounds[3])
axes[0].set_title("75025 + context: sidewalk surfaces and trails")
axes[1].set_title("Detail: sidewalk surfaces and parcel boundaries")
for ax in axes:
    ax.set_axis_off()
fig.suptitle("EmPath · Cleaned infrastructure geometry", fontsize=17)
fig.text(.05, .025, "Plano public GIS, retrieved November 8, 2026. Orange: sidewalk surfaces; teal: trail lines. Connectivity not validated.", fontsize=10)
fig.tight_layout(rect=[0, .05, 1, .95])
OUT = ROOT / "output/data_cleaning"
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "cleaned_infrastructure.png", dpi=170, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(OUT / "cleaned_infrastructure.png")))

## Behavioral training observations

All original control/treatment rows are retained when they pass consistency checks. The sun-present subset is separate. These are Singapore observations, not Plano labels. Preserve participant groups when splitting train/test data.

In [ ]:
choices = pd.read_csv(DATA / "singapore_choices.csv")
sun_choices = pd.read_csv(DATA / "singapore_sun_present_choices.csv")
pd.DataFrame([report["behavioral_choices"]]).T

In [ ]:
choices[["participant", "trial", "treatment", "b_chosen", "delta_length_b_minus_a", "delta_sun_length_b_minus_a"]].head(12)

## Missingness and next checks

Geometry/schema cleaning is complete for these inputs. Raster, weather, traffic, OSM, crossing validity and residential classification still need their own checks.

In [ ]:
missing = pd.DataFrame({name: layer["missing_attributes"] for name, layer in report["layers"].items()})
missing.fillna("not in layer")